# Stop means stop: end-to-end run

Distills a scripted teacher into **Tiny Aya Global** with **AgentTune**, curates the data with **CuratorKIT**, and scores everything with **AuditKIT** on a deterministic booking simulator. Then pushes the adapter, data, eval logs and a replay Space to Hugging Face.

**Before you run**
1. Kaggle: *Settings → Accelerator → GPU T4 x2* (not P100: current PyTorch builds dropped it). *Internet → On*.
2. *Add-ons → Secrets*: add `HF_TOKEN` (a **write** token).
3. Accept the licence on https://huggingface.co/CohereLabs/tiny-aya-global with the same HF account.
4. Push the repo to your GitHub and set `REPO_URL` below.

**Run `SMOKE = True` first** (about 20 minutes, catches setup problems), then set `SMOKE = False` and *Run all* again (about 3 to 3.5 hours on T4 x2 with the second seed; about 2.5 hours without).

Works on Colab too (single GPU: the parallel steps just run one after another).

In [ ]:
REPO_URL = "https://github.com/bruce249/stop-means-stop"   # <- your repo
HF_USER  = "your-hf-username"                              # <- your HF username
SMOKE    = True    # True: tiny end-to-end check. False: the real run.
RUN_CONTROL = True # also train the no-interrupt control adapter (adds ~40 min, runs in parallel on GPU 1)
SECOND_SEED = True # retrain the main adapter with a second seed (adds ~45 min). Turn off if short on time.
PUSH     = not SMOKE

In [ ]:
import os, subprocess, sys, time, json, pathlib
ON_KAGGLE = os.path.exists("/kaggle/working")
WORK = "/kaggle/working" if ON_KAGGLE else "/content"
REPO = f"{WORK}/stop-means-stop"
if ON_KAGGLE:
    from kaggle_secrets import UserSecretsClient
    os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
else:
    from google.colab import userdata
    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
if not os.path.exists(REPO):
    subprocess.run(["git", "clone", REPO_URL, REPO], check=True)
os.chdir(REPO)
print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv"], capture_output=True, text=True).stdout)

## 1. Environment (about 6 minutes)
AgentTune 1.1.4 needs Python 3.12 and torch 2.11, so everything runs in its own `uv` virtualenv, whatever the notebook's own Python is.

In [ ]:
%%bash
set -e
cd /kaggle/working/stop-means-stop 2>/dev/null || cd /content/stop-means-stop
pip install -q uv
uv venv --python 3.12 .venv -q
ok=0
for idx in cu126 cu128; do
  if uv pip install -q --python .venv "torch==2.11.0" torchvision --index-url https://download.pytorch.org/whl/$idx; then ok=1; echo "torch from $idx"; break; fi
done
[ $ok = 1 ] || uv pip install -q --python .venv "torch==2.11.0" torchvision
uv pip install -q --python .venv -r requirements.txt
.venv/bin/python -c "import torch; print(torch.__version__, torch.cuda.is_available(), torch.cuda.device_count()); x=torch.randn(64,64,device='cuda'); print('gpu matmul ok', (x@x).shape)"
.venv/bin/python -c "import agenttune, auditkit, curatorkit, trl, transformers, peft; print('agenttune', agenttune.__version__, '| trl', trl.__version__, '| transformers', transformers.__version__)"

In [ ]:
PY = f"{REPO}/.venv/bin/python"
N_GPU = int(subprocess.run([PY, "-c", "import torch; print(torch.cuda.device_count())"], capture_output=True, text=True).stdout.strip() or 1)

def run(args, log, gpu=None, background=False):
    env = dict(os.environ)
    if gpu is not None:
        env["CUDA_VISIBLE_DEVICES"] = str(gpu)
    f = open(f"{REPO}/runs/{log}", "w")
    p = subprocess.Popen([PY] + args, cwd=REPO, env=env, stdout=f, stderr=subprocess.STDOUT)
    if background:
        return p
    rc = p.wait()
    print(open(f"{REPO}/runs/{log}").read()[-2500:])
    if rc != 0:
        raise RuntimeError(f"{args[0]} failed, see runs/{log}")

def wait(procs):
    while any(p.poll() is None for p in procs.values()):
        time.sleep(60)
        print(time.strftime("%H:%M"), {k: ("running" if p.poll() is None else f"exit {p.returncode}") for k, p in procs.items()})
    for k, p in procs.items():
        print(f"--- {k}: exit {p.returncode}\n", open(f"{REPO}/runs/{k}.log").read()[-1500:])
        if p.returncode != 0:
            raise RuntimeError(f"{k} failed")

os.makedirs(f"{REPO}/runs", exist_ok=True)
print("GPUs:", N_GPU)

## 2. Data: scenarios → scripted teacher → CuratorKIT → AgentTune EventLogs (1 minute, CPU)

In [ ]:
run(["scripts/make_data.py", "--out", "data"] + (["--scale", "0.1"] if SMOKE else []), "make_data.log")
LIMIT = ["--limit", "24"] if SMOKE else []
run(["scripts/evaluate.py", "--policy", "oracle", "--name", "oracle"] + LIMIT, "eval-oracle.log")
run(["scripts/evaluate.py", "--policy", "never_stop", "--name", "never_stop"] + LIMIT, "eval-never_stop.log")

## 3. Baseline eval (GPU 1) in parallel with training the main adapter (GPU 0)
Real run: about 60 to 75 minutes on T4. Progress is printed every minute; full logs are in `runs/`.

In [ ]:
TRAIN_X = ["--max-examples", "60"] if SMOKE else []
g1 = 1 if N_GPU > 1 else 0
procs = {"eval-base": run(["scripts/evaluate.py", "--policy", "hf", "--name", "base"] + LIMIT, "eval-base.log", gpu=g1, background=True)}
if N_GPU == 1:
    wait(procs); procs = {}
procs["train-main"] = run(["scripts/train.py", "--split", "main", "--out", "runs/adapter-main"] + TRAIN_X, "train-main.log", gpu=0, background=True)
wait(procs)

## 4. Tuned eval (GPU 0) in parallel with the control adapter (GPU 1)
The control adapter uses the same recipe on episodes **without** interrupts. If it does not stop, the stop behaviour comes from the interrupt data, not from format learning.

In [ ]:
procs = {"eval-tuned": run(["scripts/evaluate.py", "--policy", "hf", "--name", "tuned", "--adapter", "runs/adapter-main"] + LIMIT,
                           "eval-tuned.log", gpu=0, background=True)}
if RUN_CONTROL:
    if N_GPU == 1:
        wait(procs); procs = {}
    procs["train-control"] = run(["scripts/train.py", "--split", "control_no_interrupts", "--out", "runs/adapter-control"] + TRAIN_X,
                                 "train-control.log", gpu=g1, background=True)
wait(procs)
if RUN_CONTROL:
    run(["scripts/evaluate.py", "--policy", "hf", "--name", "control", "--adapter", "runs/adapter-control"] + LIMIT, "eval-control.log", gpu=g1)

## 4b. Second seed (optional)
Same recipe, different seed. Shows the result is not a lucky run.

In [ ]:
if SECOND_SEED:
    run(["scripts/train.py", "--split", "main", "--seed", "7", "--out", "runs/adapter-s2"] + TRAIN_X, "train-s2.log", gpu=0)
    run(["scripts/evaluate.py", "--policy", "hf", "--name", "tuned_s2", "--adapter", "runs/adapter-s2"] + LIMIT, "eval-tuned_s2.log", gpu=0)

## 5. Results

In [ ]:
run(["scripts/report.py", "--runs", "runs"], "report.log")
from IPython.display import Image, Markdown, display
display(Image(f"{REPO}/runs/results.png"))
display(Markdown(open(f"{REPO}/runs/results.md").read()))

## 6. Publish to Hugging Face (public, un-gated) and build the two collections

In [ ]:
if PUSH:
    run(["scripts/push_to_hub.py", "--user", HF_USER, "--github", REPO_URL], "push.log")
    print(json.dumps(json.load(open(f"{REPO}/runs/hub_links.json")), indent=2))
else:
    print("SMOKE run: skipping the push. Set SMOKE = False and run all again.")

In [ ]:
# Download everything (adapters excluded) as one zip, for the README and article numbers.
import shutil
tmp = f"{WORK}/sms-results"
shutil.rmtree(tmp, ignore_errors=True)
shutil.copytree(f"{REPO}/runs", tmp, ignore=shutil.ignore_patterns("adapter-*"))
shutil.make_archive(f"{WORK}/stop-means-stop-results", "zip", tmp)
print(f"{WORK}/stop-means-stop-results.zip")